In this notebook, we aim to take a pre-trained model from hugging face using [Whisper](https://huggingface.co/openai/whisper-large-v2). The most recent pre-trained model for ASR from OpenAI. And its also accompanied by an [article on arxiv](https://arxiv.org/pdf/2212.04356.pdf) published in december 2022. 

(we will also try using [Wav2Vec2](https://ai.facebook.com/blog/wav2vec-20-learning-the-structure-of-speech-from-raw-audio/) and  [XLSR-Wav2Vec2](https://ai.facebook.com/blog/-xlm-r-state-of-the-art-cross-lingual-understanding-through-self-supervision/) :
[Wav2Vec2-XLS-R-300M](https://huggingface.co/facebook/wav2vec2-xls-r-300m)
, [Wav2Vec2-XLS-R-1B](https://huggingface.co/facebook/wav2vec2-xls-r-1b)
and [Wav2Vec2-XLS-R-2B](https://huggingface.co/facebook/wav2vec2-xls-r-2b). )
 
 
and fine-tuning with [swahili data](https://huggingface.co/datasets/mozilla-foundation/common_voice_11_0) from mozilla common voice hosted in hugging face dataset platform. 


## Install all the requirements

In [ ]:
!nvidia-smi
!pip install datasets
!pip install transformers==4.11.3
!pip install torchaudio==0.10.0+cu113 -f https://download.pytorch.org/whl/cu113/torch_stable.html #Install version 0.10.0 with CUDA support for NVIDIA GPUs.
!pip install jiwer

Tue Feb 21 05:33:44 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 510.47.03    Driver Version: 510.47.03    CUDA Version: 11.6     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |
| N/A   62C    P0    29W /  70W |      0MiB / 15360MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

 We will use notebook_login() function to access token which we then use to authenticate to the Hugging Face Hub and allow us to download datasets,  models, and save our checkpoints during training. The Git Large File Storage (LFS) package will help us upload our model checkpoints:

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

Token is valid.
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [ ]:
!apt install git-lfs

Reading package lists... Done
Building dependency tree       
Reading state information... Done
git-lfs is already the newest version (2.9.2-1).
The following package was automatically installed and is no longer required:
  libnvidia-common-510
Use 'apt autoremove' to remove it.
0 upgraded, 0 newly installed, 0 to remove and 21 not upgraded.


## Data

In this stage, we download the common voice data, and the prepare it for fine-tuning one of the three pre-trained models we mentioned at the beginning.

In [ ]:
from datasets import load_dataset

training_data = load_dataset("mozilla-foundation/common_voice_11_0", "sw", split="train[:10%]")
testing_data = load_dataset("mozilla-foundation/common_voice_11_0", "sw", split="test[:5%]")

Computing checksums:   8%|8         | 1/12 [00:06<01:11,  6.54s/it]

Extracting data files:   0%|          | 0/5 [00:00<?, ?it/s]

Extracting data files:   0%|          | 0/5 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]


Reading metadata...: 0it [00:00, ?it/s]
Reading metadata...: 10388it [00:00, 103869.20it/s]
Reading metadata...: 26614it [00:00, 115665.31it/s]


...

Generating other split: 0 examples [00:00, ? examples/s]




Reading metadata...: 0it [00:00, ?it/s]


Reading metadata...: 7688it [00:00, 76870.73it/s]


Reading metadata...: 15738it [00:00, 79001.43it/s]


Reading metadata...: 23639it [00:00, 78709.17it/s]


Reading metadata...: 31563it [00:00, 78914.87it/s]


Reading metadata...: 39547it [00:00, 79244.74it/s]


Reading metadata...: 47472it [00:00, 79087.34it/s]


Reading metadata...: 55525it [00:00, 79553.90it/s]


Reading metadata...: 63481it [00:00, 79545.85it/s]


Reading metadata...: 71436it [00:00, 78355.66it/s]


Reading metadata...: 79311it [00:01, 78473.21it/s]


Reading metadata...: 87279it [00:01, 78837.65it/s]


Reading metadata...: 95165it [00:01, 77652.67it/s]


Reading metadata...: 102936it [00:01, 76769.65it/s]


Reading metadata...: 110618it [00:01, 75623.21it/s]


Reading metadata...: 118186it [00:01, 75346.42it/s]


Reading metadata...: 125921it [00:01, 75936.73it/s]


Reading metadata...: 133604it [00:01, 76193.18it/s]


Reading metadata...: 141255it [00:01, 76281.65it/s

Generating invalidated split: 0 examples [00:00, ? examples/s]


Reading metadata...: 0it [00:00, ?it/s]
Reading metadata...: 10400it [00:00, 103992.41it/s]
Reading metadata...: 24710it [00:00, 126989.67it/s]
Reading metadata...: 47470it [00:00, 131442.43it/s]


Dataset common_voice_11_0 downloaded and prepared to /root/.cache/huggingface/datasets/mozilla-foundation___common_voice_11_0/sw/11.0.0/2c65b95d99ca879b1b1074ea197b65e0497848fd697fdb0582e0f6b75b6f4da0. Subsequent calls will reuse this data.


In [ ]:
training_data

Dataset({
    features: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment'],
    num_rows: 2661
})

In [ ]:
testing_data

Dataset({
    features: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment'],
    num_rows: 512
})

###  We observe that:

For training data, we have 11 columns : `['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment']`, and `26614` rows. 

For testing data, we have the same number of columns but now `10238` rows.
    

###  Let try exploring the data:

1. Remove the unrequired columns from both training and testing set
2. Then output 10 random sentences from the trainin set

In [ ]:
#we only remain with the path, audio and sentence which are the only columns the model will require for training
training_data = training_data.remove_columns(["accent", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"])
testing_data = testing_data.remove_columns(["accent", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"])

In [ ]:
# we only have the path, audio and sentence as we expected
print(training_data)
print(testing_data)

Dataset({
    features: ['path', 'audio', 'sentence'],
    num_rows: 2661
})
Dataset({
    features: ['path', 'audio', 'sentence'],
    num_rows: 512
})


In [ ]:
#we now generate 10 random sentences from our two datasets
import random
import pandas as pd
from IPython.display import display, HTML
from datasets import ClassLabel

#this function will receive a dataset then output 10 random sentences

def display_random_elements(dataset, num_examples=10):

  #we first confirm that the dataset is more than 10 sentences
    if num_examples > len(dataset):
        raise ValueError("Can't pick more elements than there are in the dataset.")

  # returns a list of unique, randomly selected integers from 0 to dataset-1
    random_indices = random.sample(range(len(dataset)), num_examples)

    # converts the picked examples into a Pandas DataFrame and displays
    df = pd.DataFrame(dataset[random_indices])
    display(HTML(df.to_html()))

In [ ]:
# lets see any 10 examples on the training set
display_random_elements(training_data.remove_columns(["path", "audio"]), num_examples=10)

,sentence
0,Hivyo waliwafukuza kwa kupigana kama ilivyokuwa kwa makabila mengi
1,alianzisha mikakati mbalimbali ya kimaendeleo kuimarisha uchumi
2,Vimepita vizazi na vizazi vya wanadamu
3,Hali zilizozingira kuzuka kwa vita hazieleweki
4,Tatizo lililowakumba walowezi wa Liberia lilitokana na chuki ya viongozi wa nchi za Kiafrika
5,wanasiasa wakubwa kutoka kaskazini wakitaka mwislamu kupokea kijiti hicho
6,Hivi ni virusi vipya vya korona ambavyo havijatambuliwa hapo awali kwa wanadamu
7,forodhani ni eneo zuri kwa watalii kujifunza vyakula vya asili ya zanzibar
8,wanajiita Wahadzabe akimaanisha watu katika lugha yao Hadzane
9,Wanyaturu hao walikaa hapo kidogo na ndio eneo hilo ambalo watu hao waligawanyika tena


In [ ]:
#lets also see 10 from testing set
display_random_elements(testing_data.remove_columns(["path", "audio"]), num_examples=10)

,sentence
0,Yapo maneno ambayo mmeyaswahilisha wenyewe.
1,Rungwe unazidiwa urefu wa mita moja tu na Mlima Mtorwi
2,Castro alikuwa mmoja wa wafungwa kadhaa waliofikishwa mahakamani
3,Fatuma halidumba kuolegwa.
4,Anatazama mpira uwanjani
5,Alizaliwa wa tisa kati ya wana.
6,Ziwa Edward liko katika mwinuko wa mita mia tisa na ishirini
7,Hata ivyo wananchi wa pande zote mbili ndio wanao teseka
8,Kifo mara nyingi husababishwa na maudhi au uchovu
9,Tanzania ilijibu kwa njia ya vita na jeshi lake


Let's extract all distinct letters of the training and test data and build our vocabulary from this set of letters.

In [ ]:
# take a batch of sentences
def extract_all_chars(batch):

  # Concatenates all the sentences in the batch into a single string, separating each sentence with a space character
  all_text = " ".join(batch["sentence"])

  # we remove any duplicates from the sentences 
  vocab = list(set(all_text))

  # we then return a list of unique characters
  return {"vocab": [vocab], "all_text": [all_text]}

In [ ]:
# we use map function to both the entire training and testing data
training_vocabulary = training_data.map(extract_all_chars, batched=True, batch_size=-1, remove_columns= training_data.column_names )
testing_vocabulary = testing_data.map(extract_all_chars, batched=True, batch_size=-1, remove_columns = testing_data.column_names)

  0%|          | 0/1 [00:00<?, ?ba/s]

  0%|          | 0/1 [00:00<?, ?ba/s]

In [ ]:
#lets also see 10 from testing set ( this is test!!)
display_random_elements(testing_data.remove_columns(["path", "audio"]), num_examples=10)

,sentence
0,Hakumbuki habari ya kwanza aliyoiandika alipoingia Mwananchi
1,Polio ni ugonjwa wa mfumo mkuu wa neva.
2,Kinachotoa maamuzi ya kisera katika muundo wa Benki Kuu
3,Kwenza Nazi mbili na ndimu moja
4,Mwana ndhuri ni ule apendae kuridhiya vadhadhi vake.
5,Kutokana na ripoti hiyo iliyotolewa na shirika la kuwahudumia wakimbizi duniani
6,Hachufungi idirisha la mwisho.
7,Kasha hili limerithiwa vizazi vitatu sasa
8,Hifadhi za wanyama huhitaji taratibu hizi kama mkataba wa kuchukua.
9,Wilson Mukama aliwahi kuwa Katibu Mkuu wa Chama cha mapinduzi


In [ ]:
print(training_vocabulary)
print(testing_vocabulary)

Dataset({
    features: ['vocab', 'all_text'],
    num_rows: 1
})
Dataset({
    features: ['vocab', 'all_text'],
    num_rows: 1
})


We will now create a list of all the unique letters found in both the training and test datasets, and then creating a dictionary where each unique letter is assigned a numerical value 

In [ ]:
# we create a new list of unique elements from the training and testing vocabulary list
vocabulary_list = list(set(training_vocabulary["vocab"][0]) | set(testing_vocabulary["vocab"][0]))

# then we create a dictionary of the unique elements and their count
vocabulary_dict = {cha: i for i, cha in enumerate(sorted(vocabulary_list))}
vocabulary_dict


{' ': 0,
 '"': 1,
 "'": 2,
 '*': 3,
 ',': 4,
 '-': 5,
 '.': 6,
 '/': 7,
 ':': 8,
 ';': 9,
 '?': 10,
 'A': 11,
 'B': 12,
 'C': 13,
 'D': 14,
 'E': 15,
 'F': 16,
 'G': 17,
 'H': 18,
 'I': 19,
 'J': 20,
 'K': 21,
 'L': 22,
 'M': 23,
 'N': 24,
 'O': 25,
 'P': 26,
 'Q': 27,
 'R': 28,
 'S': 29,
 'T': 30,
 'U': 31,
 'V': 32,
 'W': 33,
 'Y': 34,
 'Z': 35,
 'a': 36,
 'b': 37,
 'c': 38,
 'd': 39,
 'e': 40,
 'f': 41,
 'g': 42,
 'h': 43,
 'i': 44,
 'j': 45,
 'k': 46,
 'l': 47,
 'm': 48,
 'n': 49,
 'o': 50,
 'p': 51,
 'q': 52,
 'r': 53,
 's': 54,
 't': 55,
 'u': 56,
 'v': 57,
 'w': 58,
 'x': 59,
 'y': 60,
 'z': 61,
 'á': 62,
 'â': 63,
 'é': 64,
 '‘': 65,
 '’': 66}

In [ ]:
#lets also see 10 from testing set (this is also a test!!)
display_random_elements(testing_data.remove_columns(["path", "audio"]), num_examples=10)

,sentence
0,Karangiza mchuzi wa nyama
1,Iliyotuma silaha na vifaa vingine kwa jeshi lake
2,Ali ameumwa na mbuvo mpaka amepekwa sipichali
3,Na kupata matokeo mazuri ya dawa hizo
4,Maradona hakuwa kocha aliyefanya vizuri
5,hifadhi ni maarufu kwa wanyama jamii ya tumbili wajulikanao kama mbega weusi na weupe
6,Sogeza gari mbele.
7,Simba alimfunga Yanga magoli matano
8,tanzania imebarikiwa kuwa na maziwa Kama victoria nyasa tanganyika na mengine mengi
9,Anajulikana sana kwa kasi yake na uwezo wa kiufundi.


From above code, we see we have 89 characters including special characters and both capital and small letters

We can remove special characters that do not change the pronounciation of words. As we can see above, characters such as ":",".", etc

We also don't want the model to think that "R" and "r" have different pronunciation. So we can convert all the characters into lower case

In [ ]:
import re
chars_to_remove_regex = '[\,\?\.\!\-\;\:\"\“\%\‘\”\?\'\…\•\°\(\)\=\*\/\`\ː\’]'

def remove_special_characters(batch):
    batch["sentence"] = re.sub(chars_to_remove_regex, '', batch["sentence"]).lower()
    return batch

In [ ]:
# we map the "remove_special_characters" function on both the trainnng and testing data

training_data = training_data.map(remove_special_characters)
testing_data = testing_data.map(remove_special_characters)


  0%|          | 0/2661 [00:00<?, ?ex/s]

  0%|          | 0/512 [00:00<?, ?ex/s]

In [ ]:
#lets also see 10 from testing set (this is a test!!!)
display_random_elements(testing_data.remove_columns(["path", "audio"]), num_examples=10)

,sentence
0,uchumi wa mongolia hutegemea hasa migodi na ufugaji
1,wakiongozwa na aliyekuwa kiogozi wa chama cha ukombozi wa kenya jommo kenyatta
2,kasha hili limerithiwa vizazi vitatu sasa
3,ama kwa hakika hakuna kinachofanana na upendo wa mama
4,watu waliamua kupaita eneo hilo kwa mdigo
5,mlima kenya ni pili mlima wa juu katika afrika
6,nyama yao ni nzuri sana
7,joto limezidi sasa december itakuaje
8,umoja na mshikamano ni nyenzo bora ya kujenga taifa
9,jitihadi haiondoi kudura


Let substitute the characters with hatted characters. In swahili, we really don't have characters such as "ū", "ó","á", etc. We will assume that they meant "u", "o","a", etc. So we just want to convert all those characters with those special marks into their close substitutions.

In [ ]:
def replace_hatted_characters(batch):
    sentence = batch["sentence"]
    sentence = re.sub('\s([â, á, å, ã])\s', ' a ', sentence)
    sentence = re.sub('\s([î, ï])\s', ' i ', sentence)
    sentence = re.sub('\s([ô])\s', ' o ', sentence)
    sentence = re.sub('\s([š])\s', ' s ', sentence)
    sentence = re.sub('\s([ñ])\s', ' n ', sentence)
    sentence = re.sub('\s([é])\s', ' e ', sentence)
    sentence = re.sub('\s([û,ū,ụ,ú,µ])\s', ' u ', sentence)
    sentence = re.sub('\s([ø,ö,ó])\s', ' o ', sentence)
    batch["sentence"] = sentence
    return batch

In [ ]:
training_data = training_data.map(replace_hatted_characters)
testing_data = testing_data.map(replace_hatted_characters)

  0%|          | 0/2661 [00:00<?, ?ex/s]

  0%|          | 0/512 [00:00<?, ?ex/s]

In [ ]:
#lets also see 10 from testing set (this is a test!!!)
display_random_elements(testing_data.remove_columns(["path", "audio"]), num_examples=10)

,sentence
0,kunakumbuka tuyosoma chuoni
1,usichomboe ito langu
2,kridinal adam stefan sapieah alikuwa askofu mkuu wa kanisa katoriki
3,chuoni nilianzisha kikundi cha ufundishaji teknolojia
4,kasha hili limerithiwa vizazi vitatu sasa
5,mwaka mzima kwenye msitu ni kijani tupu kama vile mtu anamwagili
6,maporomoko ya kimani yanapatikana ndani ya pori la akiba la mpanga kipengere
7,kuna njia nyingi za kufanya hivi lakini mchakato kamili kwa kawaida huitwa kushikilia uchaguzi
8,dalili a mvua ni mawingu
9,hata mtoto akukosee kiasi gani kamwe huwezi kuruhusu kosa kukutawala


Lets re-run the function that extracts the characters from both training and testing set (It's above), then run the code block below to extract a new vocabulary list.  See whether we have removed the special characters and whether all the letters are lower case

In [ ]:
# we create a new list of unique elements from the training and testing vocabulary list
vocabulary_list = list(set(training_vocabulary["vocab"][0]) | set(testing_vocabulary["vocab"][0]))

# then we create a dictionary of the unique elements and their count
vocabulary_dict = {cha: i for i, cha in enumerate(sorted(vocabulary_list))}
vocabulary_dict

{' ': 0,
 'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26,
 'á': 27,
 'â': 28,
 'é': 29}

We can see now we have only lower case alphabet and space (" "). These are the only characters the model needs to learn. 

Now we give the space (" ") a token "|" as that is a requirement in Connectionist Temporal Classification algorithm.

We only add the padding token and unknown token.

In [ ]:
vocabulary_dict["|"] = vocabulary_dict[" "]
del vocabulary_dict[" "]

In [ ]:
vocabulary_dict["[UNK]"] = len(vocabulary_dict)
vocabulary_dict["[PAD]"] = len(vocabulary_dict)
len(vocabulary_dict)

32

We now save the vocabulary into a file. We will name that file as vocab_file

In [ ]:
import json
with open('vocab.json', 'w') as vocab_file:
    json.dump(vocabulary_dict, vocab_file)

In [ ]:
# lets see how the training set looks like now
display_random_elements(training_data.remove_columns(["path", "audio"]), num_examples=5)


# and lets also see how the testing set looks like now
display_random_elements(testing_data.remove_columns(["path", "audio"]), num_examples=5)

,sentence
0,isikushangaze kati ya hizo siku sita au nane utakazo kuwa unapanda mlima hutaweza kuoga
1,simu langu bado linanisumbua
2,wakati mwingine hata haiwezekani na wavulana lazima
3,kumsemele mwezio
4,utando wa uyoga ukishatanda vizuri weka mifuko kwenye sehemu ya kuoteshea kama vile kichanja


,sentence
0,la urusi tuvans khakass altai chuvash wana jimbo
1,hii ni kwa sababu ya tofauti inayopatikana katika kuhesabu silabi
2,uzito una uwiano sawa na masi
3,haikuwa rahisi kwa siwa kushughulikia kazi za hapa na pale za nyumbani kwao
4,siku hizi hutumia gurudumu au plastiki kuyatengeneza


we load the vocabulary to [wav2vecCTC tokenizer](https://huggingface.co/docs/transformers/v4.26.1/en/main_classes/tokenizer#transformers.PreTrainedTokenizer)

In [ ]:
from transformers import Wav2Vec2CTCTokenizer

tokenizer = Wav2Vec2CTCTokenizer("/content/vocab.json", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|")

We then create the [wav2vec feature extractor](https://huggingface.co/docs/transformers/v4.26.1/en/model_doc/wav2vec2#transformers.Wav2Vec2FeatureExtractor):
converts the raw audio to torch array which will input in the model

In [ ]:
from transformers import Wav2Vec2FeatureExtractor

feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0, do_normalize=True, return_attention_mask=True)

Wav2Vec2 processor wraps a Wav2Vec2 feature extractor and a Wav2Vec2 CTC tokenizer into a single processor. [Wav2Vec2Processor](https://huggingface.co/docs/transformers/v4.26.1/en/model_doc/wav2vec2#transformers.Wav2Vec2Processor) offers all the functionalities of Wav2Vec2FeatureExtractor and PreTrainedTokenizer. 

In [ ]:
from transformers import Wav2Vec2Processor

processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


We now can save checkpoints, processor and model on our google drive

In [ ]:
#let save the processor that we just created above

processor.save_pretrained("/content/drive/MyDrive/swahili_asr_wav2vec2_implementation")

### Audio data
- loading the swahili audio data
- resampling it and preprocessing it in accordance with the training data used to train whisper model

In [ ]:
# load the path of the first audio file of our training data
print(training_data[0]["path"])


# lets see the sampling rate and other characteristics of that audio
print(training_data[0]["audio"])

/root/.cache/huggingface/datasets/downloads/extracted/d0c515954317076cb4654c80caae844d8922490cb28ecb53f2df4b52ab7baa52/common_voice_sw_28660554.mp3
{'path': '/root/.cache/huggingface/datasets/downloads/extracted/d0c515954317076cb4654c80caae844d8922490cb28ecb53f2df4b52ab7baa52/common_voice_sw_28660554.mp3', 'array': array([ 0.        ,  0.        ,  0.        , ..., -0.00168494,
       -0.00197235, -0.00124964], dtype=float32), 'sampling_rate': 48000}


We notice that the audio is mp3 and already sampled at 16KHZ.

In [ ]:
#we use the Audio object from hugging face datasets function
from datasets import Audio

#we resample the training and testing data to 16KHz as that is the sampling rate used to train the model
training_data = training_data.cast_column("audio", Audio(sampling_rate=16000))
testing_data = testing_data.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
#lets see the first columns

print(training_data[0]["audio"])
print(testing_data[0]["audio"])

{'path': '/root/.cache/huggingface/datasets/downloads/extracted/d0c515954317076cb4654c80caae844d8922490cb28ecb53f2df4b52ab7baa52/common_voice_sw_28660554.mp3', 'array': array([ 0.        ,  0.        ,  0.        , ..., -0.00162212,
       -0.00171909, -0.00184702], dtype=float32), 'sampling_rate': 16000}
{'path': '/root/.cache/huggingface/datasets/downloads/extracted/51be8e6185f5507509f311a8e86535649a659f2e388d898f86a8a45ad84306fd/common_voice_sw_31428161.mp3', 'array': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32), 'sampling_rate': 16000}


We can listen to a random clips

In [ ]:
import IPython.display as ipd
import numpy as np
import random

# Choose a random integer between 0 and the number of examples in the dataset (exclusive)
rand_index = random.randint(0, len(training_data) - 1)

# Print the sentence at the chosen random index from the dataset
sentence = training_data[rand_index]["sentence"]
print(sentence)

# Play the audio at the chosen random index from the dataset
audio = training_data[rand_index]["audio"]["array"]
ipd.Audio(data=audio, autoplay=True, rate=16000)


wataalamu wanaeleza kuwa vilima hivi vya barafu vina umri wa kuanzia miaka elfu moja


In [ ]:
rand_int = random.randint(0, len(training_data)-1)

print("Target text:", training_data[rand_int]["sentence"])
print("Input array shape:", training_data[rand_int]["audio"]["array"].shape)
print("Sampling rate:", training_data[rand_int]["audio"]["sampling_rate"])

Target text: yanga itapaswa kusubiri wachezaji wazoeane
Input array shape: (70272,)
Sampling rate: 16000


the data is 1-dimensional array, the sampling rate is 16kHz, and the target text is normalized (But it seems like we removed the spaces between words, which we shall correct before fine tuning it on a model)


So going forward, we complete the data preparation by passing the  wav2vec processor to transform for training. 

In [ ]:
def process_dataset(batch):
    # Retrieve the audio data from the batch
    audio = batch["audio"]

    # Process the audio data using the Wav2Vec2CTCModel
    # and store the input values in the batch
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    # Store the length of the input values in the batch
    batch["input_length"] = len(batch["input_values"])
    
    # Using the Wav2Vec2CTCModel as a target processor
    with processor.as_target_processor():
        # Process the sentence and store the input IDs in the batch
        batch["labels"] = processor(batch["sentence"]).input_ids
        
    # Return the modified batch
    return batch


The process_data function is passed on all examples in the training and testing sets. The Wav2Vec2Processor only normalizes the data.

In [ ]:
training_data = training_data.map(process_dataset, remove_columns = training_data.column_names)
testing_data = testing_data.map(process_dataset, remove_columns = testing_data.column_names)

  0%|          | 0/2661 [00:00<?, ?ex/s]

  0%|          | 0/512 [00:00<?, ?ex/s]

Training: Lets set up a [trainer](https://huggingface.co/docs/transformers/main/main_classes/trainer). The Trainer class provides an API for feature-complete training in PyTorch for most standard use cases.

1. DataCollatorCTCWithPadding(), is used for padding input sequences and labels to the same length. * :obj:`True` or :obj:`'longest'`: Pad to the longest sequence in the batch (or no padding if only a single sequence if provided) This is the strategy we'll use.

The collactor function is inspired by the [huggingface/transformer run_speech_recognition_ctc.py](https://github.com/huggingface/transformers/blob/7e61d56a45c19284cfda0cee8995fb552f6b1f4e/examples/pytorch/speech-recognition/run_speech_recognition_ctc.py#L219) script on GitHub.

In [ ]:
import torch

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union

# dataclass generator generates special methods for a class, such as __init__, __repr__, and __eq__, based on the class variables.
@dataclass
class DataCollatorCTCWithPadding:

    processor: Wav2Vec2Processor

    # padding method used for the input sequences and defaults to True.
    padding: Union[bool, str] = True

    # the function takes in a list of features, where each feature is a dictionary containing input values and labels,
    # and returns a dictionary containing the padded input values and labels.
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        
        # extracts the input values from the features.
        input_features = [{"input_values": feature["input_values"]} for feature in features]

        # extracts the labels from the features.
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        # pads the input sequences to the same length then return the result as PyTorch tensors.
        batch = self.processor.pad(input_features,padding=self.padding,return_tensors="pt", )

        
        with self.processor.as_target_processor():
          # pads the labels to the same length as the input sequences 
            labels_batch = self.processor.pad(label_features,padding=self.padding,return_tensors="pt",)

        # replaces padding in the labels with -100 so that it is ignored when calculating the loss.
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        #  sets the padded labels as the "labels" key in the batch dictionary.
        batch["labels"] = labels

        return batch

**Note:** A data collator is a function or class that takes a list of samples from a dataset and combines them into batches to feed into a machine learning model. The data collator can apply padding or truncation to ensure that the sequences within each batch have the same length. It can also apply any necessary data pre-processing steps, such as tokenization or numerical encoding. The purpose of the data collator is to ensure that the model receives input data in a format that it can process efficiently.

In [ ]:
# define the data collator
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

We now load the pretrained checkpoint of [Wav2Vec2-XLS-R-300M](https://huggingface.co/facebook/wav2vec2-xls-r-300m).

The hyperparameters we choose is quite random (and inspired by [Patrick's blog](https://huggingface.co/blog/fine-tune-xlsr-wav2vec2)): We will play with the parameters to find better results.


In [ ]:
from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-300m", 
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.0,
    ctc_loss_reduction="mean", 
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)

Downloading:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.18G [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/wav2vec2-xls-r-300m were not used when initializing Wav2Vec2ForCTC: ['quantizer.weight_proj.weight', 'project_hid.weight', 'project_q.bias', 'project_q.weight', 'project_hid.bias', 'quantizer.codevectors', 'quantizer.weight_proj.bias']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-xls-r-300m and are newly initialized: ['lm_head.weight', 'lm_head.bias']
You should probably TRAIN this model on a down-stream task to be able to use it 

We are freezing the layers of the model that were already trained. We only want to add a CTC loss on top of the transformer without trying to fine tune the CNN part of the architecture.

In [ ]:
model.freeze_feature_extractor()

TrainingArguments accesses all the points of customization during training.

In our case, we specify where the checkpoints will be stored, the evaluation strategy, learning rate and so on.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
  output_dir= "/content/drive/MyDrive/swahili_asr_wav2vec2_implementation",
  group_by_length=True,
  per_device_train_batch_size=16,
  gradient_accumulation_steps=2,
  evaluation_strategy="steps",
  num_train_epochs=15,
  gradient_checkpointing=True,
  fp16=True,
  save_steps=200,
  eval_steps=200,
  logging_steps=400,
  learning_rate=3e-4,
  warmup_steps=500,
  save_total_limit=2,
  push_to_hub=False,
)

We now create an evaluation:

1. we need the evaluate library
2. then import the downloaded library
3. We will then define the metric as Word Error Rate (WER).
4. we will finaly define the metric function ( it will basically computes the Word Error Rate (WER) metric between the predicted and true labels for a given batch of examples) that we shall eventually pass it on to out trainer.

In [ ]:
!pip install evaluate nltk rouge_score

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.4/81.4 KB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24955 sha256=9a2bd92e4eccbd53bb55ecbfa5c557c8044550aee24802797baece94bc2d2313
  Stored in directory: /root/.cache/pip/wheels/24/55/6f/ebfc4cb176d1c9665da4e306e1705496206d08215c1acd9dde
Successfully built rouge_score


In [ ]:
import evaluate

In [ ]:
metric = evaluate.load("wer")


This is the function that computes the Word Error Rate (WER).

In [ ]:
def compute_metrics(pred):
    # Get the predicted logits from the model's output.
    pred_logits = pred.predictions

    # Get the predicted token ids by taking the index with maximum probability across the last dimension of the logits tensor.
    pred_ids = np.argmax(pred_logits, axis=-1)
 
    # we replace the -100 pad with corresponding padding id
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    # Convert the predicted token ids to string
    pred_str = processor.batch_decode(pred_ids)
   

    # Convert the true label token ids to string
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    # Compute the WER metric between the predicted and true label strings 
    wer = metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

The trainer will take in the instances of the functions we had instanciated in different variables.

They are:
1. the model -> that is the Wav2Vec2ForCTC, which has the pretrained XLS-R wav2vec 2.0 model with 300m parameters.

2. the data collator -> that converts inputs into the required format to be inputs in a machine learning model. 

3. THe training arguments -> that contains the specifics of the model we are training, like the directory we store the model, learning rate among other hyperparameters, which we shall be playing with inorder to get the best model.

4. The metric function -> that computes the word error rate between the predicted string and the true label string.

5. the training and testing sets with allow for training and testing of the model.The testing data in this case is evaluating the performance of the model.

6. The tokenizer ->  used during training, evaluation, and inference to ensure that the input data is properly processed and tokenized in the same way. (The tokenizer is responsible for converting the input data into tokens that can be used as input to the model)

refence on [hugging face](https://huggingface.co/docs/evaluate/transformers_integrations)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=training_data,
    eval_dataset=testing_data,
    tokenizer=processor.feature_extractor,
)

Using amp fp16 backend


We then call the train function on our trainer. Fine tuning just began!

In [ ]:
trainer.train()

The following columns in the training set  don't have a corresponding argument in `Wav2Vec2ForCTC.forward` and have been ignored: input_length.
***** Running training *****
  Num examples = 2661
  Num Epochs = 15
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 2
  Total optimization steps = 1245
/usr/local/lib/python3.8/dist-packages/transformers/feature_extraction_utils.py:158: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at  ../torch/csrc/utils/tensor_new.cpp:201.)
  tensor = as_tensor(value)
/usr/local/lib/python3.8/dist-packages/transformers/models/wav2vec2/modeling_wav2vec2.py:882: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like t

Step,Training Loss,Validation Loss,Wer
200,No log,2.930781,1.000000
400,3.576100,0.934445,0.706388
600,3.576100,0.878459,0.580720
800,0.377700,0.835381,0.528455


The following columns in the evaluation set  don't have a corresponding argument in `Wav2Vec2ForCTC.forward` and have been ignored: input_length.
***** Running Evaluation *****
  Num examples = 512
  Batch size = 8
Saving model checkpoint to /content/drive/MyDrive/swahili_asr_wav2vec2_implementation/checkpoint-200
Configuration saved in /content/drive/MyDrive/swahili_asr_wav2vec2_implementation/checkpoint-200/config.json
Model weights saved in /content/drive/MyDrive/swahili_asr_wav2vec2_implementation/checkpoint-200/pytorch_model.bin
Configuration saved in /content/drive/MyDrive/swahili_asr_wav2vec2_implementation/checkpoint-200/preprocessor_config.json
/usr/local/lib/python3.8/dist-packages/transformers/models/wav2vec2/modeling_wav2vec2.py:882: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the curr

Step,Training Loss,Validation Loss,Wer
200,No log,2.930781,1.000000
400,3.576100,0.934445,0.706388
600,3.576100,0.878459,0.580720
800,0.377700,0.835381,0.528455
1000,0.377700,0.786289,0.489199
1200,0.133300,0.830490,0.470616


The following columns in the evaluation set  don't have a corresponding argument in `Wav2Vec2ForCTC.forward` and have been ignored: input_length.
***** Running Evaluation *****
  Num examples = 512
  Batch size = 8
Saving model checkpoint to /content/drive/MyDrive/swahili_asr_wav2vec2_implementation/checkpoint-1000
Configuration saved in /content/drive/MyDrive/swahili_asr_wav2vec2_implementation/checkpoint-1000/config.json
Model weights saved in /content/drive/MyDrive/swahili_asr_wav2vec2_implementation/checkpoint-1000/pytorch_model.bin
Configuration saved in /content/drive/MyDrive/swahili_asr_wav2vec2_implementation/checkpoint-1000/preprocessor_config.json
Deleting older checkpoint [/content/drive/MyDrive/swahili_asr_wav2vec2_implementation/checkpoint-600] due to args.save_total_limit
/usr/local/lib/python3.8/dist-packages/transformers/models/wav2vec2/modeling_wav2vec2.py:882: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It curr

TrainOutput(global_step=1245, training_loss=1.316315149973674, metrics={'train_runtime': 7651.9342, 'train_samples_per_second': 5.216, 'train_steps_per_second': 0.163, 'total_flos': 6.650920234233999e+18, 'train_loss': 1.316315149973674, 'epoch': 14.99})

In [ ]:
checkpoints_path = "/content/drive/MyDrive/swahili_asr_wav2vec2_implementation"

In [ ]:
model = Wav2Vec2ForCTC.from_pretrained(checkpoints_path).to("cuda")


loading configuration file /content/drive/MyDrive/swahili_asr_wav2vec2_implementation/checkpoint-1200/config.json
Model config Wav2Vec2Config {
  "_name_or_path": "facebook/wav2vec2-xls-r-300m",
  "activation_dropout": 0.0,
  "apply_spec_augment": true,
  "architectures": [
    "Wav2Vec2ForCTC"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "classifier_proj_size": 256,
  "codevector_dim": 768,
  "contrastive_logits_temperature": 0.1,
  "conv_bias": true,
  "conv_dim": [
    512,
    512,
    512,
    512,
    512,
    512,
    512
  ],
  "conv_kernel": [
    10,
    3,
    3,
    3,
    3,
    2,
    2
  ],
  "conv_stride": [
    5,
    2,
    2,
    2,
    2,
    2,
    2
  ],
  "ctc_loss_reduction": "mean",
  "ctc_zero_infinity": false,
  "diversity_loss_weight": 0.1,
  "do_stable_layer_norm": true,
  "eos_token_id": 2,
  "feat_extract_activation": "gelu",
  "feat_extract_dropout": 0.0,
  "feat_extract_norm": "layer",
  "feat_proj_dropout": 0.0,
  "feat_quantizer_dropout": 0

Trying to get some predictions from the testing data

In [ ]:
input_dict = processor(testing_data[2]["input_values"], sampling_rate = 16000, return_tensors="pt", padding=True)

logits = model(input_dict.input_values.to("cuda")).logits

pred_ids = torch.argmax(logits, dim=-1)[0]

In [ ]:
print(processor.decode(pred_ids))

inajulikana kama shina la waranji


We are loading the test set again to compare the predictions from the model versus the ground truth.

In [ ]:
transcription = load_dataset("mozilla-foundation/common_voice_11_0", "sw", split="test[:3%]")

In [ ]:
transcription[3]['sentence'].lower()

'inajulikana kama shina la warangi.'

We want to create an interface for our model.
- we want to use gradio and pipeline from hugging face

Since our model is saved in our drive, we will push it to hugging face hub then try creating the interface using our model.

In [ ]:
!pip install gradio

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 42.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 KB 8.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 KB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.9/56.9 KB 7.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.0/107.0 KB 13.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 KB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.5/84.5 KB 11.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.7/140.7 KB 16.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 67.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 KB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━